# Notebook 1 — Data Availability Detection & Extraction
**When the Floor Speaks, Does the Market Listen?**
*Employee Sentiment, Stock Performance, and the Dual-Level Quality Signal*

Yulin Tian & Ilia Kornietskii | BI Norwegian Business School | GRA 19701 Master Thesis

---

**Purpose:** Identify firms with sufficient Glassdoor review coverage for the US-direction analysis. Detect data availability across 87 months (January 2016 to March 2023) and extract the final firm universe.

### Notebook setup (paths + quick peek)

In [1]:
import re
from pathlib import Path
import pandas as pd
import numpy as np

# Paths 
ALL_REVIEWS_PATH = Path(r"C:\Users\tiany\Downloads\all_reviews.csv")
#TARGETS_PATH     = Path(r"C:\Users\tiany\Downloads\target_companies_2016_2025.csv")

OUT_DIR   = Path(r"C:\Users\tiany\Downloads\all_reviews_targets_export")
LOG_DIR   = OUT_DIR / "_logs"
OUT_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)

# Read targets (84 firms)
# targets = pd.read_csv(TARGETS_PATH)
# print("Targets shape:", targets.shape)
# display(targets.head(10))

# TARGET_NAME_COL = "company"  
# print("Target name column candidates:", list(targets.columns))

In [2]:
# Quick peek of all_reviews (columns + first rows)
df_preview = pd.read_csv(ALL_REVIEWS_PATH, nrows=20, low_memory=False)
print("all_reviews columns:\n", list(df_preview.columns))
display(df_preview.head(5))

all_reviews columns:
 ['rating', 'title', 'status', 'pros', 'cons', 'advice', 'Recommend', 'CEO Approval', 'Business Outlook', 'Career Opportunities', 'Compensation and Benefits', 'Senior Management', 'Work/Life Balance', 'Culture & Values', 'Diversity & Inclusion', 'firm_link', 'date', 'job', 'index']


,rating,title,status,pros,cons,advice,Recommend,CEO Approval,Business Outlook,Career Opportunities,Compensation and Benefits,Senior Management,Work/Life Balance,Culture & Values,Diversity & Inclusion,firm_link,date,job,index
0,5.0,Good,"Current Employee, more than 10 years",Knowledge gain of complete project,Financial growth and personal growth,NaN,v,o,v,3.0,3.0,3.0,3.0,3.0,3.0,Reviews/Baja-Steel-and-Fence-Reviews-E5462645.htm,"Nov 19, 2022",Manager Design,NaN
1,4.0,Good,"Former Employee, less than 1 year","Good work,good work , flexible, support","Good,work, flexible,good support, good team work",NaN,v,o,o,4.0,4.0,4.0,4.0,4.0,4.0,Reviews/Baja-Steel-and-Fence-Reviews-E5462645.htm,"Jan 29, 2022",Anonymous Employee,NaN
2,4.0,"Supervising the manufacturing the processes, e...","Current Employee, more than 1 year",This company is a best opportunity for me to l...,"Monthly Target work,Maintain production schedu...",NaN,v,o,v,2.0,3.0,2.0,2.0,2.0,2.0,Reviews/Baja-Steel-and-Fence-Reviews-E5462645.htm,"Aug 12, 2021",Production Engineer,NaN
3,1.0,terrible,"Current Employee, more than 1 year",I wish there were some to list,too many to list here,NaN,x,x,x,1.0,3.0,1.0,3.0,1.0,NaN,https://www.glassdoor.com/Reviews/Calgary-Flam...,"Sep 24, 2020",Senior Account Executive,NaN
4,4.0,"It could be so good, but it isn’t","Current Employee, more than 3 years",Fast Paced. Endless challenges. Inclusive envi...,The biggest perk of the job provides no value ...,NaN,o,o,o,3.0,3.0,3.0,1.0,4.0,5.0,https://www.glassdoor.com/Reviews/Calgary-Flam...,"Mar 25, 2023",Assistant Manager,NaN


### 1) Derive 'Company Name' from firm_link

In [3]:
def extract_company_name(link: str):
    """
    Extract a readable company name from Glassdoor firm_link.

    Works for:
    - Reviews/Baja-Steel-and-Fence-Reviews-E5462645.htm
    - https://www.glassdoor.com/Reviews/Some-Company-Reviews-E123456.htm

    Returns: "Baja Steel and Fence" (or None if not matched)
    """
    if pd.isna(link):
        return None

    s = str(link).strip()

    patterns = [
        r"(?:https?://[^/]+)?/Reviews/([^/]+?)-Reviews-E\d+",  # full URL
        r"^Reviews/([^/]+?)-Reviews-E\d+",                     # relative path
    ]

    slug = None
    for p in patterns:
        m = re.search(p, s)
        if m:
            slug = m.group(1)
            break

    if not slug:
        return None

    return slug.replace("-", " ").strip()


In [4]:
# 1.1 Load full dataset and create Company Name
df = pd.read_csv(ALL_REVIEWS_PATH, low_memory=False)

df["Company Name"] = df["firm_link"].apply(extract_company_name)

print("Rows:", len(df))
print("Company Name non-null %:", df["Company Name"].notna().mean())
display(df[["firm_link", "Company Name"]].head(10))

Rows: 9901889
Company Name non-null %: 0.9997142969386952


,firm_link,Company Name
0,Reviews/Baja-Steel-and-Fence-Reviews-E5462645.htm,Baja Steel and Fence
1,Reviews/Baja-Steel-and-Fence-Reviews-E5462645.htm,Baja Steel and Fence
2,Reviews/Baja-Steel-and-Fence-Reviews-E5462645.htm,Baja Steel and Fence
3,https://www.glassdoor.com/Reviews/Calgary-Flam...,Calgary Flames
4,https://www.glassdoor.com/Reviews/Calgary-Flam...,Calgary Flames
5,https://www.glassdoor.com/Reviews/Calgary-Flam...,Calgary Flames
6,https://www.glassdoor.com/Reviews/Calgary-Flam...,Calgary Flames
7,https://www.glassdoor.com/Reviews/Calgary-Flam...,Calgary Flames
8,https://www.glassdoor.com/Reviews/Calgary-Flam...,Calgary Flames
9,https://www.glassdoor.com/Reviews/Calgary-Flam...,Calgary Flames


### 2) Parse review date and create year field (2016–2025 focus)

In [5]:
# Parse date
DATE_COL = "date"  

df["review_date"] = pd.to_datetime(df[DATE_COL], errors="coerce")
df["review_year"] = df["review_date"].dt.year

print("Parsed review_date non-null %:", df["review_date"].notna().mean())
display(df[[DATE_COL, "review_date", "review_year", "Company Name"]].head(10))

Parsed review_date non-null %: 0.9999827305678745


,date,review_date,review_year,Company Name
0,"Nov 19, 2022",2022-11-19,2022.0,Baja Steel and Fence
1,"Jan 29, 2022",2022-01-29,2022.0,Baja Steel and Fence
2,"Aug 12, 2021",2021-08-12,2021.0,Baja Steel and Fence
3,"Sep 24, 2020",2020-09-24,2020.0,Calgary Flames
4,"Mar 25, 2023",2023-03-25,2023.0,Calgary Flames
5,"Apr 6, 2023",2023-04-06,2023.0,Calgary Flames
6,"Oct 29, 2022",2022-10-29,2022.0,Calgary Flames
7,"Feb 2, 2023",2023-02-02,2023.0,Calgary Flames
8,"Jan 17, 2023",2023-01-17,2023.0,Calgary Flames
9,"Aug 17, 2022",2022-08-17,2022.0,Calgary Flames


In [6]:
# Basic data quality checks
# How many rows have BOTH Company Name and review_date parsed?
valid_mask = df["Company Name"].notna() & df["review_date"].notna()
print("Valid rows (Company+Date):", int(valid_mask.sum()), "out of", len(df))

# Year distribution quick check 
year_counts = df.loc[valid_mask, "review_year"].value_counts().sort_index()
display(year_counts.head(15))
display(year_counts.tail(15))

Valid rows (Company+Date): 9898889 out of 9901889


review_year
2008.0      38342
2009.0      42898
2010.0      81796
2011.0     104342
2012.0     192117
2013.0     243908
2014.0     404838
2015.0     667224
2016.0     748273
2017.0     760445
2018.0     684043
2019.0     721505
2020.0     963420
2021.0    1959304
2022.0    1796040
Name: count, dtype: int64

review_year
2009.0      42898
2010.0      81796
2011.0     104342
2012.0     192117
2013.0     243908
2014.0     404838
2015.0     667224
2016.0     748273
2017.0     760445
2018.0     684043
2019.0     721505
2020.0     963420
2021.0    1959304
2022.0    1796040
2023.0     490394
Name: count, dtype: int64

### 3) Existence test (manual name-variant search)

In [7]:
def normalize_text(s: str) -> str:
    if pd.isna(s):
        return ""
    s = str(s).lower().strip()
    s = re.sub(r"[^\w\s]", " ", s)     # remove punctuation
    s = re.sub(r"\s+", " ", s).strip() # normalize whitespace
    return s

def existence_test(df: pd.DataFrame, query: str, top_n: int = 30) -> pd.DataFrame:
    """
    Search company candidates by substring match on normalized Company Name.
    Returns top candidates by total review volume.
    """
    qn = normalize_text(query)
    name_norm = df["Company Name"].fillna("").map(normalize_text)

    hits = (
        df.loc[name_norm.str.contains(qn, na=False), "Company Name"]
          .value_counts()
          .head(top_n)
          .reset_index()
    )
    hits.columns = ["Company Name (dataset)", "n_reviews_total"]
    return hits


In [8]:

# Example of the firm name usage checking
existence_test(df, "bmw", top_n=20)

# BMW Group	1607

,Company Name (dataset),n_reviews_total
0,BMW Group,1607
1,BMW of North America,266
2,BMW Financial Services,225
3,BMW of San Francisco,29
4,Budds' BMW,26
5,Crevier BMW,11
6,Partridge Hampshire BMW,7
7,Wide World BMW,7
8,BMW of Austin,6
9,BMW Foundation Herbert Quandt,1


In [10]:
# # STEP 1: Load dataset
# # =========================================================
# df = pd.read_csv(r"C:\Users\tiany\Downloads\all_reviews.csv", low_memory=False)


# # STEP 2: Derive company name from firm_link
# # (Uses my previously defined extract_company_name function)
# # =========================================================
# if "Company Name" not in df.columns:
#     df["Company Name"] = df["firm_link"].apply(extract_company_name)

# # Basic cleaning
# df["Company Name"] = df["Company Name"].astype("string").str.strip()
# df = df[df["Company Name"].notna() & (df["Company Name"] != "")].copy()


# # STEP 3: Parse date and extract year
# # =========================================================
# df["review_date"] = pd.to_datetime(df["date"], errors="coerce")
# df["review_year"] = df["review_date"].dt.year


# # STEP 4: Total review count per firm (entire dataset)
# # =========================================================
# total_counts = (
#     df.groupby("Company Name")
#       .size()
#       .reset_index(name="Total Reviews")
# )

# # STEP 5: Yearly review count per firm for 2016–2023
# # =========================================================
# years_needed = list(range(2016, 2024))

# yearly_counts = (
#     df[df["review_year"].isin(years_needed)]
#     .groupby(["Company Name", "review_year"])
#     .size()
#     .unstack(fill_value=0)
# )

# # Ensure every year 2016-2023 exists as a column
# yearly_counts = yearly_counts.reindex(columns=years_needed, fill_value=0)

# # Convert year columns to string names for CSV output
# yearly_counts.columns = [str(y) for y in yearly_counts.columns]
# yearly_counts = yearly_counts.reset_index()

# # STEP 6: Merge total counts with yearly counts
# # =========================================================
# firm_review_summary = total_counts.merge(
#     yearly_counts,
#     on="Company Name",
#     how="left"
# )

# # Fill missing yearly values with 0
# year_cols = [str(y) for y in years_needed]

# for y in year_cols:
#     if y not in firm_review_summary.columns:
#         firm_review_summary[y] = 0

# firm_review_summary[year_cols] = firm_review_summary[year_cols].fillna(0).astype(int)

# # STEP 7: Rank firms from highest total reviews to lowest
# # =========================================================
# firm_review_summary = firm_review_summary.sort_values(
#     by=["Total Reviews", "Company Name"],
#     ascending=[False, True]
# ).reset_index(drop=True)

# # STEP 8: Reorder columns exactly as required
# # First column: firm name
# # Second column: total reviews
# # Third onward: 2016 ... 2023
# # =========================================================
# final_cols = ["Company Name", "Total Reviews"] + [str(y) for y in years_needed]
# firm_review_summary = firm_review_summary[final_cols]

# # STEP 9: Export to the required folder
# # =========================================================
# output_dir = Path(r"C:\Users\tiany\OneDrive\GRA1971 Master Thesis\Data Resources")
# output_dir.mkdir(parents=True, exist_ok=True)

# output_file = output_dir / "all_unique_firms_review_ranking_2016_2023.csv"
# firm_review_summary.to_csv(output_file, index=False, encoding="utf-8-sig")

# print("Done.")
# print("Output saved to:", output_file)
# print("Number of unique firms:", len(firm_review_summary))

# # Optional preview
# display(firm_review_summary.head(20))

Done.
Output saved to: C:\Users\tiany\OneDrive\GRA1971 Master Thesis\Data Resources\all_unique_firms_review_ranking_2016_2023.csv
Number of unique firms: 34317


,Company Name,Total Reviews,2016,2017,2018,2019,2020,2021,2022,2023
0,Amazon,163396,4312,6956,7901,8612,16274,39007,57350,16153
1,Tata Consultancy Services,107218,5415,6162,6880,8910,11288,26541,25085,4445
2,Walmart,102152,7069,7933,6561,6626,10531,22821,21267,3129
3,Cognizant Technology Solutions,84171,4615,5666,5915,7385,9722,25494,14030,1739
4,McDonald s,76777,3919,4278,4005,4944,9235,23303,16729,2394
5,Accenture,69026,4204,4021,4341,5253,8675,18408,11846,2699
6,Target,67885,4961,4603,3776,3811,5650,14127,13983,2150
7,HP Inc,63787,3486,4371,6628,7238,7858,15931,12405,5451
8,Starbucks,55325,3710,3847,3667,4174,6924,12768,9497,2336
9,Infosys,53189,3416,3354,3336,3917,6818,13922,7833,1456


### 4) Monthly Review Availability Check — 87 Target Firms (2016 Jan – 2023 Dec)

In [ ]:
import calendar

# ═══════════════════════════════════════════════════════════════════════════════
# SECTION 4 — Monthly Review Counts for 87 Target Firms  (2016 Jan → 2023 Dec)
#
# Relies on:
#   • extract_company_name()  — defined in Section 1
#   • existence_test()        — defined in Section 3
#   • df                      — loaded & pre-processed in Sections 1–2

# ── Guard: reload df if it has not been set up yet ────────────────────────────
if "Company Name" not in df.columns or "review_date" not in df.columns:
    df = pd.read_csv(r"C:\Users\tiany\Downloads\all_reviews.csv", low_memory=False)
    df["Company Name"] = df["firm_link"].apply(extract_company_name)
    df["Company Name"] = df["Company Name"].astype("string").str.strip()
    df = df[df["Company Name"].notna() & (df["Company Name"] != "")].copy()
    df["review_date"] = pd.to_datetime(df["date"], errors="coerce")
    df["review_year"]  = df["review_date"].dt.year
    print("df reloaded inside Section 4 guard.")

# ── 4.1  Target firm names (87 firms — taken directly from first column) ──────
TARGET_FIRMS = [
    "Amazon",
    "Walmart",
    "Cognizant Technology Solutions",
    "McDonald s",
    "Accenture",
    "Target",
    "HP Inc",
    "Starbucks",
    "Infosys",
    "IBM",
    "Wipro",
    "Oracle",
    "The Home Depot",
    "General Motors GM",
    "General Motors (GM)",
    "Wells Fargo",
    "Boeing",
    "Verizon",
    "Microsoft",
    "Bank of America",
    "Best Buy",
    "ADP",
    "Apple",
    "CVS Health",
    "Honeywell",
    "UPS",
    "Google",
    "Citi",
    "Cisco Systems",
    "HSBC",
    "GE",
    "Intel Corporation",
    "Lockheed Martin",
    "SAP",
    "Kroger",
    "Genpact",
    "FedEx",
    "TD",
    "Qualcomm",
    "Deutsche Bank",
    "Chipotle",
    "Dollar Tree",
    "Nokia",
    "Barclays",
    "Marriott International",
    "Comcast",
    "Salesforce",
    "Costco Wholesale",
    "Ericsson Worldwide",
    "Morgan Stanley",
    "Goldman Sachs",
    "PayPal",
    "Vodafone",
    "The Coca Cola Company",
    "Abbott",
    "American Express",
    "RBC",
    "Raytheon Technologies",
    "PepsiCo",
    "Capital One",
    "Spectrum",
    "American Airlines",
    "CGI",
    "Dollar General",
    "ICICI Bank",
    "T Mobile",
    "NIKE",
    "Manpower",
    "Ulta Beauty",
    "IQVIA",
    "Shell",
    "Amdocs",
    "UBS",
    "BNY Mellon",
    "U S Bank",
    "FIS",
    "Allstate",
    "Lowe's Home Improvement",
    "Macy's",
    "Wendy's",
    "Domino's",
    "Kohl's",
    "AT&T",
    "Procter & Gamble",
    "Johnson & Johnson",
    "J.P. Morgan",
    "Hewlett Packard Enterprise HPE",
]
print(f"Target firms loaded: {len(TARGET_FIRMS)}")

# ── 4.2  Filter dataset to target firms, restrict to 2016-2023 ───────────────
target_set = set(TARGET_FIRMS)

df_tgt = df[
    df["Company Name"].isin(target_set) &
    df["review_date"].notna()
].copy()

df_tgt["review_year"]  = df_tgt["review_date"].dt.year
df_tgt["review_month"] = df_tgt["review_date"].dt.month
df_tgt = df_tgt[
    (df_tgt["review_year"] >= 2016) &
    (df_tgt["review_year"] <= 2023)
]

print(f"Rows in 2016-2023 for target firms: {len(df_tgt):,}")

# Quick check — show which target firms have zero matches (name may differ in dataset)
found_names = set(df_tgt["Company Name"].unique())
missing = [nm for nm in TARGET_FIRMS if nm not in found_names]
if missing:
    print(f"\nWARNING: {len(missing)} firm(s) returned 0 rows — names may differ in dataset:")
    for nm in missing:
        print(f"  '{nm}'")
        print("  Closest matches in dataset:")
        display(existence_test(df, nm.split()[0].lower(), top_n=5))
else:
    print("All firms found in the dataset.")


Target firms loaded: 87
Rows in 2016-2023 for target firms: 1,905,953
All firms found in the dataset.


In [10]:
  # ── 4.2b  Consolidate duplicate firm names ────────────────────────────────
  CONSOLIDATE = {
      "General Motors GM":   "General Motors",
      "General Motors (GM)": "General Motors",
  }
  df_tgt["Company Name"] = df_tgt["Company Name"].replace(CONSOLIDATE)
  df["Company Name"]     = df["Company Name"].replace(CONSOLIDATE)   # keeps total_reviews in sync

  # Also update TARGET_FIRMS so the output row order stays clean
  TARGET_FIRMS_CLEAN = []
  seen = set()
  for nm in TARGET_FIRMS:
      canonical = CONSOLIDATE.get(nm, nm)
      if canonical not in seen:
          TARGET_FIRMS_CLEAN.append(canonical)
          seen.add(canonical)
  TARGET_FIRMS = TARGET_FIRMS_CLEAN
  print(f"Firms after consolidation: {len(TARGET_FIRMS)}")   # should be 86

Firms after consolidation: 86


In [11]:
# ── 4.3  Monthly review counts per firm ──────────────────────────────────────
monthly_counts = (
    df_tgt
    .groupby(["Company Name", "review_year", "review_month"])
    .size()
    .reset_index(name="cnt")
)

# Total reviews per firm across ALL years (not restricted to 2016-2023)
total_reviews = (
    df[df["Company Name"].isin(target_set)]
    .groupby("Company Name")
    .size()
)

# 96 month-column labels: '2016 Jan', '2016 Feb', ..., '2023 Dec'
month_labels = [
    f"{yr} {calendar.month_abbr[mo]}"
    for yr in range(2016, 2024)
    for mo in range(1, 13)
]

# ── 4.4  Build output DataFrame (one row per firm, preserving order) ──────────
rows = []
for firm_nm in TARGET_FIRMS:
    total   = int(total_reviews.get(firm_nm, 0))
    row     = {"Company Name": firm_nm, "Total Reviews": total}
    firm_mc = monthly_counts[monthly_counts["Company Name"] == firm_nm]
    for yr in range(2016, 2024):
        for mo in range(1, 13):
            label = f"{yr} {calendar.month_abbr[mo]}"
            mask  = (firm_mc["review_year"] == yr) & (firm_mc["review_month"] == mo)
            row[label] = int(firm_mc.loc[mask, "cnt"].iloc[0]) if mask.any() else 0
    rows.append(row)

df_out = pd.DataFrame(rows, columns=["Company Name", "Total Reviews"] + month_labels)

print(f"\nOutput shape: {df_out.shape}")
display(df_out[["Company Name", "Total Reviews",
                "2016 Jan", "2016 Feb", "2023 Nov", "2023 Dec"]].head(10))

# ── 4.5  Export CSV ───────────────────────────────────────────────────────────
out_dir = Path(r"C:\Users\tiany\OneDrive\GRA1971 Master Thesis\Data Resources")
out_dir.mkdir(parents=True, exist_ok=True)

out_csv = out_dir / "86firms_monthly_review_counts_2016_2023.csv"
df_out.to_csv(out_csv, index=False, encoding="utf-8-sig")

print(f"\nDone!  CSV saved to:\n  {out_csv}")
months_with_data = int((df_out[month_labels] > 0).sum().sum())
print(f"Firms: {len(df_out)}  |  Months: 96  |  Month-firm cells with reviews: {months_with_data}")


Output shape: (86, 98)


,Company Name,Total Reviews,2016 Jan,2016 Feb,2023 Nov,2023 Dec
0,Amazon,163396,377,362,0,0
1,Walmart,102152,728,588,0,0
2,Cognizant Technology Solutions,84171,401,378,0,0
3,McDonald s,76777,398,347,0,0
4,Accenture,69026,400,439,0,0
5,Target,67885,544,406,0,0
6,HP Inc,63787,247,280,0,0
7,Starbucks,55325,350,290,0,0
8,Infosys,53189,285,341,0,0
9,IBM,52072,268,258,0,0



Done!  CSV saved to:
  C:\Users\tiany\OneDrive\GRA1971 Master Thesis\Data Resources\86firms_monthly_review_counts_2016_2023.csv
Firms: 86  |  Months: 96  |  Month-firm cells with reviews: 7515


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# SECTION 5 — Final Analysis Dataset  (86 firms, 2016 Jan – 2023 Mar)
#
# • Keeps ALL original columns from df (no columns dropped or renamed).
# • Both "General Motors GM" and "General Motors (GM)" are included;
#   their Company Name was consolidated to "General Motors" in Section 4,
#   so all their reviews travel together under that single label.
# • Expected row count: 1,898,445

ANALYSIS_START = pd.Timestamp("2016-01-01")
ANALYSIS_END   = pd.Timestamp("2023-03-31")

# TARGET_FIRMS is the 86-firm list built at the end of Section 4 (post-consolidation)
target_set_final = set(TARGET_FIRMS)   # 86 firms, "General Motors" covers both GM slugs

df_final = df[
    df["Company Name"].isin(target_set_final) &
    df["review_date"].notna() &
    (df["review_date"] >= ANALYSIS_START) &
    (df["review_date"] <= ANALYSIS_END)
].copy()

print(f"Rows in final dataset : {len(df_final):,}")        # expect 1,898,445
print(f"Unique firms          : {df_final['Company Name'].nunique()}")  # expect 86
print(f"Date range            : {df_final['review_date'].min().date()}  →  {df_final['review_date'].max().date()}")
print(f"Columns kept          : {list(df_final.columns)}")

# ── Save CSV ──────────────────────────────────────────────────────────────────
out_dir   = Path(r"C:\Users\tiany\OneDrive\GRA1971 Master Thesis\Data Resources")
out_dir.mkdir(parents=True, exist_ok=True)

out_csv = out_dir / "86firms_reviews_2016jan_2023mar.csv"
df_final.to_csv(out_csv, index=False, encoding="utf-8-sig")

print(f"\nSaved: {out_csv}")

Rows in final dataset : 1,898,445
Unique firms          : 86
Date range            : 2016-01-01  →  2023-03-31
Columns kept          : ['rating', 'title', 'status', 'pros', 'cons', 'advice', 'Recommend', 'CEO Approval', 'Business Outlook', 'Career Opportunities', 'Compensation and Benefits', 'Senior Management', 'Work/Life Balance', 'Culture & Values', 'Diversity & Inclusion', 'firm_link', 'date', 'job', 'index', 'Company Name', 'review_date', 'review_year']

Saved: C:\Users\tiany\OneDrive\GRA1971 Master Thesis\Data Resources\86firms_reviews_2016jan_2023mar.csv
